In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("Social_Network_Ads.csv")
df.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [4]:
prob = df['Purchased']


In [5]:
#this can fail if one class is not available
# total = prob.count()
# total_zero = prob.value_counts()[0]
# total_one = prob.value_counts()[1]

# zero_prob = total_zero/total
# one_prob = total_one/total

# print (zero_prob)
# print (one_prob)

# df.iloc[:, -1].value_counts(normalize=True)

In [6]:
def find_probability(column):
    total = column.count()
    counts = column.value_counts()
    zero_prob = counts.get(0, 0) / total
    one_prob = counts.get(1, 0) / total
    return zero_prob, one_prob

zero_prob, one_prob = find_probability(prob)
print("P(Purchased=0) =", zero_prob)
print("P(Purchased=1) =", one_prob)


P(Purchased=0) = 0.6425
P(Purchased=1) = 0.3575


In [7]:
# Step 1: keep only the columns we want to study manually
practice_df = df[['Age', 'EstimatedSalary', 'Purchased']].copy()
practice_df.head()


,Age,EstimatedSalary,Purchased
0,19,19000,0
1,35,20000,0
2,26,43000,0
3,27,57000,0
4,19,76000,0


In [8]:
# Step 2: entropy of a target column
def entropy(column):
    probabilities = column.value_counts(normalize=True)
    return -(probabilities * np.log2(probabilities)).sum()

# np.log2(probabilities) computes: [log2(0.6425), log2(0.3575)]
# probabilities * np.log2(probabilities) computes: [p1 log2(p1), p2 log2(p2)]

parent_entropy = entropy(practice_df['Purchased'])
print('Entropy of Purchased =', parent_entropy)


Entropy of Purchased = 0.9405884140193839


In [12]:
# Step 3: entropy after splitting on a threshold
def split_entropy(dataframe, feature, threshold, target='Purchased'):
    left = dataframe[dataframe[feature] <= threshold]
    right = dataframe[dataframe[feature] > threshold]

    if len(left) == 0 or len(right) == 0:
        return float('inf')

    left_weight = len(left) / len(dataframe)
    right_weight = len(right) / len(dataframe)

    return left_weight * entropy(left[target]) + right_weight * entropy(right[target])

#testing with a random value
age_threshold = 42.5
weighted_entropy = split_entropy(practice_df, 'Age', age_threshold)
information_gain = parent_entropy - weighted_entropy

print('Weighted entropy after split =', weighted_entropy)
print('Information gain =', information_gain)


Weighted entropy after split = 0.6342832603357957
Information gain = 0.3063051536835881


In [18]:
# Step 4: check each Age value and keep the one with the lowest entropy
best_age = None
lowest_entropy_age = float('inf')

for salary_value in sorted(practice_df['Age'].unique()):
    current_entropy_age = split_entropy(practice_df, 'Age', salary_value)

    if current_entropy_age < lowest_entropy_age:
        lowest_entropy_age = current_entropy_age
        best_age = salary_value

information_gain_age = entropy(practice_df['Purchased']) - lowest_entropy_age

print('Best Age threshold:', best_age)
print('Lowest entropy:', lowest_entropy_age)
print('Information gain:', information_gain_age)


Best Age threshold: 42
Lowest entropy: 0.6342832603357957
Information gain: 0.3063051536835881


In [16]:
# Step 5: check each Estimated Salary and keep the one with the lowest entropy
best_salary = None
lowest_entropy_salary = float('inf')

for salary_value in sorted(practice_df['EstimatedSalary'].unique()):
    current_entropy_salary = split_entropy(practice_df, 'EstimatedSalary', salary_value)

    if current_entropy_salary < lowest_entropy_salary:
        lowest_entropy_salary = current_entropy_salary
        best_salary = salary_value

information_gain_salary = entropy(practice_df['Purchased']) - lowest_entropy_salary

print('Best Salary threshold:', best_salary)
print('Lowest entropy:', lowest_entropy_salary)
print('Information gain:', information_gain_salary)

Best Salary threshold: 89000
Lowest entropy: 0.7336383271646412
Information gain: 0.2069500868547427


In [20]:
print("Age threshold:", best_age)
print("Age entropy:", lowest_entropy_age)
print("Age information gain:", information_gain_age)

print("Salary threshold:", best_salary)
print("Salary entropy:", lowest_entropy_salary)
print("Salary information gain:", information_gain_salary)

if lowest_entropy_age < lowest_entropy_salary:
    print("Root node should use: Age")
    print("Best root question: Is Age <=", best_age, "?")
else:
    print("Root node should use: EstimatedSalary")
    print("Best root question: Is EstimatedSalary <=", best_salary, "?")


Age threshold: 42
Age entropy: 0.6342832603357957
Age information gain: 0.3063051536835881
Salary threshold: 89000
Salary entropy: 0.7336383271646412
Salary information gain: 0.2069500868547427
Root node should use: Age
Best root question: Is Age <= 42 ?


In [22]:
# Step 7: split the dataset using the chosen root feature and threshold
if lowest_entropy_age < lowest_entropy_salary:
    root_feature = 'Age'
    root_threshold = best_age
else:
    root_feature = 'EstimatedSalary'
    root_threshold = best_salary

left_child = practice_df[practice_df[root_feature] <= root_threshold]
right_child = practice_df[practice_df[root_feature] > root_threshold]

print('Root feature:', root_feature)
print('Root threshold:', root_threshold)
print('Left child size:', len(left_child))
print('Right child size:', len(right_child))

print('Left child class counts:')
print(left_child['Purchased'].value_counts())

print('Right child class counts:')
print(right_child['Purchased'].value_counts())


Root feature: Age
Root threshold: 42
Left child size: 285
Right child size: 115
Left child class counts:
Purchased
0    239
1     46
Name: count, dtype: int64
Right child class counts:
Purchased
1    97
0    18
Name: count, dtype: int64


In [26]:
# Step 8: check whether the left and right child nodes are pure
print('Left child unique classes:', left_child['Purchased'].unique())
print('Right child unique classes:', right_child['Purchased'].unique())

if left_child['Purchased'].nunique() == 1:
    print('Left child is pure')
else:
    print('Left child is not pure')

if right_child['Purchased'].nunique() == 1:
    print('Right child is pure')
else:
    print('Right child is not pure')


Left child unique classes: [0 1]
Right child unique classes: [1 0]
Left child is not pure
Right child is not pure


In [27]:
# Step 9: find the best Age threshold inside the left child
best_age_left = None
lowest_entropy_age_left = float('inf')

for age_value in sorted(left_child['Age'].unique()):
    current_entropy_age_left = split_entropy(left_child, 'Age', age_value)

    if current_entropy_age_left < lowest_entropy_age_left:
        lowest_entropy_age_left = current_entropy_age_left
        best_age_left = age_value

information_gain_age_left = entropy(left_child['Purchased']) - lowest_entropy_age_left

print('Best Age threshold for left child:', best_age_left)
print('Lowest Age entropy for left child:', lowest_entropy_age_left)
print('Age information gain for left child:', information_gain_age_left)


Best Age threshold for left child: 26
Lowest Age entropy for left child: 0.5710284867337677
Age information gain for left child: 0.06662815885861173


In [28]:
# Step 10: find the best EstimatedSalary threshold inside the left child
best_salary_left = None
lowest_entropy_salary_left = float('inf')

for salary_value in sorted(left_child['EstimatedSalary'].unique()):
    current_entropy_salary_left = split_entropy(left_child, 'EstimatedSalary', salary_value)

    if current_entropy_salary_left < lowest_entropy_salary_left:
        lowest_entropy_salary_left = current_entropy_salary_left
        best_salary_left = salary_value

information_gain_salary_left = entropy(left_child['Purchased']) - lowest_entropy_salary_left

print('Best Salary threshold for left child:', best_salary_left)
print('Lowest Salary entropy for left child:', lowest_entropy_salary_left)
print('Salary information gain for left child:', information_gain_salary_left)


Best Salary threshold for left child: 90000
Lowest Salary entropy for left child: 0.2920671803024909
Salary information gain for left child: 0.34558946528988854


In [29]:
# Step 11: compare the left child splits and choose its best question
print('Left child Age threshold:', best_age_left)
print('Left child Age entropy:', lowest_entropy_age_left)
print('Left child Age information gain:', information_gain_age_left)

print('Left child Salary threshold:', best_salary_left)
print('Left child Salary entropy:', lowest_entropy_salary_left)
print('Left child Salary information gain:', information_gain_salary_left)

if lowest_entropy_age_left < lowest_entropy_salary_left:
    print('Left child should split on: Age')
    print('Best left-child question: Is Age <=', best_age_left, '?')
else:
    print('Left child should split on: EstimatedSalary')
    print('Best left-child question: Is EstimatedSalary <=', best_salary_left, '?')


Left child Age threshold: 26
Left child Age entropy: 0.5710284867337677
Left child Age information gain: 0.06662815885861173
Left child Salary threshold: 90000
Left child Salary entropy: 0.2920671803024909
Left child Salary information gain: 0.34558946528988854
Left child should split on: EstimatedSalary
Best left-child question: Is EstimatedSalary <= 90000 ?
